# NEXUS synthetic demo

Run from the repository root in an environment where NEXUS is installed. This notebook uses only generated data. The risk example uses an explicitly simulated 64 MiB headroom; it is not a statement about the host machine.

In [ ]:
%load_ext nexus
%nexus verbose on

In [ ]:
import sys
from pathlib import Path

import pandas as pd

examples_dir = Path.cwd() / "examples"
if not (examples_dir / "demo_support.py").is_file():
    raise RuntimeError("Run this notebook from the repository root.")
sys.path.insert(0, str(examples_dir.resolve()))
import demo_support  # noqa: E402

safe_frame = pd.DataFrame({"key": [1, 1, 2, 2], "value": [10, 10, 20, 20]})

## Safe operation

The next cell is small. With verbose mode enabled, the pre-execution hook prints its estimate and risk classification.

In [ ]:
safe_result = safe_frame.drop_duplicates()

## Risk with a simulated resource limit

Estimate a wide join without executing it, then classify the same estimate against a deliberately small synthetic headroom. The reported drivers explain the risk.

In [ ]:
from nexus.cellpeak import cell_estimate
from nexus.pipeline import Pipeline
from nexus.risk import classify

demo_left, demo_right = demo_support.make_merge_frames()
risky_source = 'demo_left.merge(demo_right, on="key")["left_value"]'
risky_report = Pipeline(tier=0).run(risky_source, globals())
cell_est, _, _ = cell_estimate(risky_source, risky_report.estimates)
simulated = classify(
    [item.estimate for item in risky_report.estimates],
    headroom=64 * 2**20,
    cell=cell_est,
    drivers_from=risky_report.estimates,
    warn_high=cell_est.ram_high if cell_est else None,
)
print("Simulated 64 MiB headroom; host availability is not being changed.")
print(simulated.format())

## A suggestion with a re-estimate

`select_columns` is off by default because its savings were over-predicted in evaluation. Enable it explicitly for this demonstration. The suggestion engine reports its virtual Tier 0 re-estimate; it does not run the modified cell.

In [ ]:
from nexus.suggestions import suggest

suggestion_source = risky_source
suggestion_report = Pipeline(tier=0).run(suggestion_source, globals())
suggestions = suggest(
    suggestion_source,
    globals(),
    suggestion_report.inspection,
    suggestion_report.ops,
    enable={"select_columns"},
)
for item in suggestions:
    before = item.before.ram_expected / 2**20 if item.before else float("nan")
    after = item.after.ram_expected / 2**20 if item.after else float("nan")
    print(item.title)
    print(item.modified_source)
    print(f"Tier 0 predicted peak: {before:.1f} MiB -> {after:.1f} MiB")

In [ ]:
joined_value = demo_left.merge(demo_right, on="key")["left_value"]
partial_frame = pd.DataFrame({"key": [1.0, None, 2.0], "value": [3, 4, 5]})

## Partial cell

The analyzer knows `dropna`, but `value_counts` has no v4 model. Use Tier 0 for this short example so a Tier 1 statistics timeout cannot hide the known partial coverage. The pre-execution output shows an `at least` lower bound for the covered part, not a full-cell upper bound.

In [ ]:
%nexus tier 0

In [ ]:
partial_frame.dropna()
partial_frame.value_counts()

## Predicted versus measured

The estimate and actual use the same synthetic frame shape and the same operation. Actual peak RAM and elapsed time are measured in three fresh spawned processes; censored RAM remains censored rather than being treated as zero.

In [ ]:
from nexus.measure import measure_call

measure_frame = demo_support.make_measure_frame()
measure_source = "measure_frame.drop_duplicates()"
prediction = Pipeline(tier=0).run(measure_source, globals())
predicted = prediction.estimates[0].estimate
actual = measure_call(
    demo_support.drop_duplicates,
    setup=demo_support.make_measure_frame,
    repeats=3,
)
print(
    "Predicted RAM: "
    f"{predicted.ram_low / 2**20:.1f}-{predicted.ram_high / 2**20:.1f} MiB "
    f"(expected {predicted.ram_expected / 2**20:.1f} MiB)"
)
if actual.delta_bytes is None:
    print(f"Measured RAM: censored (setup bound {actual.bound_bytes / 2**20:.1f} MiB)")
else:
    print(f"Measured RAM: {actual.delta_bytes / 2**20:.1f} MiB")
print(f"Predicted time: {predicted.time_expected:.3f} s")
print(f"Measured time: {actual.time_s:.3f} s ({actual.status})")